Para acessar a API Pública, é necessário criar um aplicativo na conta do desenvolvedor principal do projeto. Para cadastrar seu aplicativo e gerar uma credencial de API, visite https://console.eduzz.com.

No menu principal, selecione a opção Meus Aplicativos e clique no botão Novo Aplicativo, então, preencha todos os campos obrigatórios e selecione os escopos que seu aplicativo irá utilizar.

ClientID: 1a607d05-9652-4690-9366-f11184985552
Secret: 683a1524414269ba7537923f83a5b94cd0e9a71d7ecd583c988cd31d76be849b

#necessário gerar url de login

#TOKEN -> apenas para teste
##em produção, usar autenticação do usuario


personal_token (https://console.eduzz.com/application/apps): edzpap_umWOviBccOeP3EjmBUdBJywCGTyhZHSmqgSSIrb5YTHdTcxQkvqLvbAeJV2aub2YocrPOszxeN85SrLx5Qb

In [8]:
# https://developers.eduzz.com/docs/api/personal-token
import requests
from datetime import datetime, timedelta

## ACESSAR: https://accounts.eduzz.com/oauth/authorize?client_id=SEU_CLIENT_ID&responseType=code&redirectTo=http://localhost:8000
# substituir "SEU_CLIENT_ID"
# fazer o login
# copiar o code da url de retorno 

#### https://accounts.eduzz.com/oauth/authorize?client_id=1a607d05-9652-4690-9366-f11184985552&responseType=code&redirectTo=http://localhost:8000
class eduzz_api():
    _instancia = None  # Controle do Singleton
    token = None       # Onde o token será guardado

    def __new__(cls):  # roda primeiro
        if cls._instancia is None:
            # Cria a instância única
            cls._instancia = super().__new__(cls)
        return cls._instancia

    def __init__(self): # roda depois do new
        # O __init__ roda toda vez que você faz api = eduzz_api()
        
        # Só autenticamos se o token ainda estiver vazio
        ####if self.token is None: self.autenticar()
        self.token   = 'edzpap__lKmRMV4dB-Aat-pMA507CwCGTyhZHSmqgSSIrb5YTE7gs5QkvqLvbAeJV2aub2YocrPOr_8cYkt5y2mL48'
        self.headers = {"Accept": "application/json", "Authorization": f"Bearer {self.token}"}

    def autenticar(self):
        print('autenticando')
        # Credenciais da Eduzz (do Console do Desenvolvedor)
        CLIENT_ID     = "1a607d05-9652-4690-9366-f11184985552"
        CLIENT_SECRET = "683a1524414269ba7537923f83a5b94cd0e9a71d7ecd583c988cd31d76be849b"
        REDIRECT_URL  = "http://localhost:8000"
        
        # colar aqui o code da url de retorno
        auth_code = "qC96I2Xm3xk1RDwoAsYqZbM2BYpXWUao76798YFW0uE"
        
        payload = {"client_id":     CLIENT_ID,
                   "client_secret": CLIENT_SECRET,
                   "code":          auth_code,
                   "redirect_uri":  REDIRECT_URL,
                   "grant_type":    "authorization_code"
                  }

        url = "https://accounts-api.eduzz.com/oauth/token"
        
        try:
            response  = requests.post(url, json=payload)
            if response.status_code == 200:
                self.data  = response.json() # dict_keys(['expires_in', 'credential', 'created_at', 'authenticated_userid', 'ttl', 'service', 'id', 'scope', 'access_token', 'refresh_token', 'token_type', 'user'])
                self.token = self.data.get('access_token')
                return self.token
            else:
                print(f"❌ Erro na API Eduzz: {response.status_code} - {response.text}")                
                return None
        except Exception as e: return None # Erro de conexão ou rede

    def api_get(self, servico, params={}):
        endpoints = {'ACCOUNTS: GERAL → Dados do Usuário':         "https://api.eduzz.com/accounts/v1/me",
                     #'ALPACLASS: ESCOLAS → Listar Minhas Escolas': "https://api.eduzz.com/alpaclass/v1/producer/teams", # params: sort
                     'BLINKET: EVENTOS → Listar Eventos':          "https://api.eduzz.com/blinket/v1/events",            # params: page, itemsPerPage, order, type, search, startDate, endDate
                     'BLINKET: MARCADORES → Listar Marcadores':    "https://api.eduzz.com/blinket/v1/attendance-tags",   # params: page, itemsPerPage
                     'MYEDUZZ: AFILIADOS → Listagem':              "https://api.eduzz.com/myeduzz/v1/affiliates",        # params: productIdsstring, pagenumber, itemsPerPagenumber
                     'MYEDUZZ: ASSINATURA → Detalhes de uma lista de assinaturas': "https://api.eduzz.com/myeduzz/v1/subscriptions", # params: page, startDate*, endDate*, filterBy (creation/update)
                     'MYEDUZZ: CLIENTES → Listagem dos Clientes':  "https://api.eduzz.com/myeduzz/v1/customers",         # params: pagenumber, itemsPerPagenumber, orderenum (asc/desc), startDatedatetime, endDatedatetime
                    }
        
        if servico=='MYEDUZZ: ASSINATURA → Detalhes de uma lista de assinaturas':
            start_date = (datetime.now() - timedelta(days=3000)).strftime("%Y-%m-%d")
            end_date   = datetime.now().strftime("%Y-%m-%d")
            params = {'startDate':start_date, 'endDate': end_date, 'filterBy':'creation'}
            
        try:
            response = requests.get(endpoints[servico], headers=self.headers,params=params)
            response.raise_for_status()           # Verifica se a requisição foi bem sucedida (status 200-299)    
            if response.status_code == 200: return response.json()
            else:                           return f"Erro {response.text}"
        except requests.exceptions.RequestException as e: print(f"Erro na requisição: {e}")


    def get_myeduzz_detalhes_cliente(self, email):
        try:
            response = requests.get(f"https://api.eduzz.com/myeduzz/v1/subscriptions/customers/{email}", headers=self.headers)
            response.raise_for_status()           # Verifica se a requisição foi bem sucedida (status 200-299)    
            if response.status_code == 200: return response.json()
            else:                           return f"Erro {response.text}"
        except requests.exceptions.RequestException as e: print(f"Erro na requisição: {e}")
        

api  = eduzz_api(); #print(api)
#api2 = eduzz_api(); print(api2)

#print(api.api_get('ACCOUNTS: GERAL → Dados do Usuário'))              # {'id': 'a1c8bc8a-...ab03', 'name': 'PLATAFORMA ARQCALC LTDA', 'email': 'contato@arqcalc.com.br'}
#### print(api.api_get('ALPACLASS: ESCOLAS → Listar Minhas Escolas'))    # Unauthorized for url: https://api.eduzz.com/alpaclass/v1/producer/teams
#print(api.api_get('BLINKET: EVENTOS → Listar Eventos'))               # VAZIO {'pages': 1, 'page': 1, 'itemsPerPage': 10, 'totalItems': 0, 'items': []}
#print(api.api_get('BLINKET: MARCADORES → Listar Marcadores'))         # VAZIO
#print(api.api_get('MYEDUZZ: AFILIADOS → Listagem'))                   # VAZIO

#dados = api.api_get('MYEDUZZ: ASSINATURA → Detalhes de uma lista de assinaturas')
#dados = api.api_get('MYEDUZZ: CLIENTES → Listagem dos Clientes')    # Demora um pouco

api.get_myeduzz_detalhes_cliente('carinemerizioarquitetura@outlook.com')   #'MYEDUZZ: ASSINATURAS → Detalhes do Cliente'

{'name': 'Carine Mariê Merizio',
 'email': 'carinemerizioarquitetura@outlook.com',
 'phone': {'countryCode': '55', 'areaCode': '47', 'number': '999412359'},
 'subscriptions': [{'id': '1658913',
   'startDate': '2022-05-20T20:46:49.000Z',
   'updatedAt': '2026-02-27T08:41:25.000Z',
   'status': 'upToDate',
   'firstInvoiceId': '39682891',
   'interruption': None,
   'isUnlimitedInstallments': False,
   'hasNegotiation': False,
   'isFinite': False,
   'recurrence': {'startsAt': '2022-05-20T20:46:49.000Z',
    'nextDueDate': '2026-03-27T00:00:00.000Z',
    'currentDueDate': '2026-02-27T00:00:00.000Z',
    'finalDueDate': '2026-02-27T00:00:00.000Z',
    'finishesAt': None,
    'frequency': {'type': 'monthly', 'value': 1}},
   'products': [{'id': '1056735',
     'name': 'Plataforma ArqCalc mensal',
     'hasMembershipFee': False,
     'membershipFee': None}],
   'payment': {'price': {'currency': 'BRL', 'value': 49.9},
    'method': 'creditCard',
    'installments': 1},
   'charges': {'tota

In [ ]:
#dados = api.api_get('MYEDUZZ: ASSINATURA → Detalhes de uma lista de assinaturas')
dados['items'][1]#.keys()

#chaves_desejadas = ['id', 'createdAt', 'updatedAt', 'status', 'isFinite']
#filtrado = {k: dados[k] for k in chaves_desejadas if k in dados}

#'id', #### id da assinatura
#'createdAt', 'updatedAt', 'status', 'firstInvoiceId', 'isFinite', 'isUnlimitedInstallments', 'hasNegotiation', 
#'interruption', 
#'recurrence': {'startsAt','nextDue','currentDue','finalDue','finishesAt','frequency':{'type','value'}},
#'charges':    {'total','current','paid','pending','negotiated'},
#'products': [ { 'id','title','hasMembershipFee','membershipFee':{'currency','value'}, 'deliveries':[] } ],
#'payment':    {'price': {'currency','value'}, 'method', 'installments'},
#'client':     {'name','email','phone':{'countryCode','areaCode','number'}}

In [6]:
#dados = api.api_get('MYEDUZZ: CLIENTES → Listagem dos Clientes')    # Demora um pouco
dados['items']

#'id', #### id do comprador
#'email',
#'phone':{'countryCode','areaCode','number'},
#'purchaseHistoryOverview': {'net': {'value', 'currency'}, 'gross':{'value','currency'}, 'productsQuantity'}    

#### gross: Total de ganhos brutos em vendas para o cliente
#### net:   Total de ganhos líquidos em vendas para o cliente

[{'id': '69778961',
  'email': 'carinemerizioarquitetura@outlook.com',
  'name': 'Carine Mariê Merizio',
  'phone': {'countryCode': '47', 'areaCode': '99', 'number': '9412359'},
  'purchaseHistoryOverview': {'net': {'value': 2043.8, 'currency': 'BRL'},
   'gross': {'value': 2195.6, 'currency': 'BRL'},
   'productsQuantity': 45}},
 {'id': '2069134965',
  'email': 'isabellecocetti.arq@gmail.com',
  'name': 'Isabelle Bertoloto Cocetti',
  'phone': {'countryCode': '19', 'areaCode': '99', 'number': '6268945'},
  'purchaseHistoryOverview': {'net': {'value': 1579.3, 'currency': 'BRL'},
   'gross': {'value': 1696.6, 'currency': 'BRL'},
   'productsQuantity': 35}},
 {'id': '903487012',
  'email': 'arquitetaeduardafurlan@gmail.com',
  'name': 'EDUARDA FURLAN DE LIMA',
  'phone': {'countryCode': '28', 'areaCode': '99', 'number': '9247196'},
  'purchaseHistoryOverview': {'net': {'value': 1393.5, 'currency': 'BRL'},
   'gross': {'value': 1497, 'currency': 'BRL'},
   'productsQuantity': 31}},
 {'id'

In [ ]:
# "https://accounts-api.eduzz.com/oauth/token"
"""
{'expires_in': 0,
 'credential': {'id': '21efc874-0064-4bbd-b085-5a0feaf1ff8b'},
 'created_at': 1772801210,
 'authenticated_userid': 'a1c8bc8a-857d-4739-acdd-89fd97a8ab03,30238211',
 'ttl': None,
 'service': None,
 'id': 'd9593860-8d7c-4d2b-86ef-465c8fd62bdf',
 'scope': 'alpaclass_delivery_read [...] webhook_write',
 'access_token': 'edzpap_UJHW6NTVfcukj-Q1QVkJIiwCGTyhZHSmqgSSIrb5YTGKr85QkvqLvbAeJV2aub2YocrPOkF78B0rCGvPIna',
 'refresh_token': None,
 'token_type': 'bearer',
 'user': {'id': 'a1c8bc8a-857d-4739-acdd-89fd97a8ab03',
          'eduzzId': 30238211,
          'nutrorId': 7709455,
          'name': 'PLATAFORMA ARQCALC LTDA',
          'email': 'contato@arqcalc.com.br'
 }}
"""

# ACCOUNTS: GERAL → Dados do Usuário GET ("https://api.eduzz.com/accounts/v1/me")
"""
{'id': 'a1c8bc8a-857d-4739-acdd-89fd97a8ab03',
 'name': 'PLATAFORMA ARQCALC LTDA',
 'email': 'contato@arqcalc.com.br'}
"""



In [ ]:
# ALPACLASS: ALUNOS → Buscar aluno                            GET  (https://api.eduzz.com/alpaclass/v1/producer/teams/{teamId}/students/{studentId})
path: teamId, studentId
# ALPACLASS: ALUNOS → Criar Aluno                             POST (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/students)
path: teamId; params: {name, email, phone, document, tags}
# ALPACLASS: ALUNOS → Documentos pendentes do aluno           PUT  (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/students/:studentId/pending-documents)
path: teamId, studentId; params: pendingDocuments
# ALPACLASS: ALUNOS → Listar alunos da escola                 GET  (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/students)
path: teamId; params (todos opcionais): page, itemsPerPage, sort, id, email, document, startDate, endDate, isFrozen
# ALPACLASS: ALUNOS → Listar matrículas do aluno              GET  (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/students/:studentId/enrollments)
path: teamId, studentId; params (todos opcionais): page, itemsPerPage

# ALPACLASS: CERTIFICADOS → Substiuir certificado por externo POST (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamSlug/enrollments/:enrollmentId/certificate)
path: teamSlug, enrollmentId; params: url

# ALPACLASS: ENTREGA → Listar cursos da entrega               GET  (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/deliveries/:deliveryId/courses)
path: teamId, deliveryId; params (todos opcionais): page, itemsPerPage, sort
# ALPACLASS: ENTREGA → Listar detalhes da entrega             GET  (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/deliveries/:deliveryId)
path: teamId, deliveryId
# ALPACLASS: ENTREGA → Listar entregas da escola              GET  (https://api.eduzz.com/alpaclass/v1/producer/teams/:teamId/deliveries)
path: teamId; params (todos opcionais): page, itemsPerPage, sort

In [ ]:
# BLINKET: EVENTOS → Detalhes do Evento GET (https://api.eduzz.com/blinket/v1/events/:id)
path: id;

# BLINKET: EVENTOS → Listar Eventos     GET (https://api.eduzz.com/blinket/v1/events)
params: page, itemsPerPage, order, type, search, startDate, endDate

# BLINKET: MARCADORES → Alterar Marcador                       PUT    (https://api.eduzz.com/blinket/v1/attendance-tags/:id)
path: id; params: title, color
# BLINKET: MARCADORES → Criar Marcador                         POST   (https://api.eduzz.com/blinket/v1/attendance-tags)
params: title, color
# BLINKET: MARCADORES → Desvincular Participantes              DELETE (https://api.eduzz.com/blinket/v1/attendance-tags/:id/attendance-list)
path: id; params: attendenceIds (array)
# BLINKET: MARCADORES → Listar Marcadores                      GET    (https://api.eduzz.com/blinket/v1/attendance-tags)
params: page, itemsPerPage
# BLINKET: MARCADORES → Listar Vínculos                        GET    (https://api.eduzz.com/blinket/v1/attendance-tags/:id/attendance-list)
path: id
# BLINKET: MARCADORES → Remover Marcador                       DELETE (https://api.eduzz.com/blinket/v1/attendance-tags/:id)
path: id
# BLINKET: MARCADORES → Vincular Participantes                 POST   (https://api.eduzz.com/blinket/v2/attendance-tags/:id/attendance-list)
path: id; params: attendanceIds (array), attendanceEmails (array)

# BLINKET: PARTCIPANTES → Adicionar Participante               POST   (https://api.eduzz.com/blinket/v1/attendance-list/add-manually)
params: eventId, ticketId, name, docuemnt, documentType, email, phone, useStock
# BLINKET: PARTCIPANTES → Alterar dados do ingresso            PUT    (https://api.eduzz.com/blinket/v1/attendance-list/update-by-invitekey/:id)
path: id; params: name, document, documentType, email, phone
# BLINKET: PARTCIPANTES → Check-in                             PUT    (https://api.eduzz.com/blinket/v1/attendance-list/check-in)
params: checkIn (bool), attendances (array), attendances[n].inviteKey
# BLINKET: PARTCIPANTES → Obter Participantes por ID do Evento GET (https://api.eduzz.com/blinket/v1/attendance-list/by-event/:id)
path: id; params: id, page, itemsPerPage, search, order (asc/desc), tickets, tags, attendanceSituation, status, modifiedAtGte, modifiedAtLle
# BLINKET: PARTCIPANTES → Reenviar Ticket                      POST (https://api.eduzz.com/blinket/v1/attendance-list/resend-ticket-email)
params: userEmail, eventId

# CHECKOUT SUN: CARRINHO → Criar um Carrinho                   POST (https://api.eduzz.com/sun/v1/cart)
params: orderId*, postbackUrl*, returnUrl, installments, shippingValue, items (array), intems[n].productId*, items[n].description*, items[n].price, items[n].price.value, items[n].price.currency, items[n].quantity*, customer, customer.email, customer.name, customer.cellphone, customer.document, customer.address, customer.address.street, customer.address.number, customer.address.neighborhood, customer.address.complement, customer.address.postalCode, customer.address.state

In [ ]:
# HEYCAMP: COMENTÁRIOS → Listar comentários                 GET    (https://api.eduzz.com/heycamp/v1/comments/on-post/:postId/community/:communityId)
path: communityId, postID

# HEYCAMP: CONVITES → Criação de Convites                   POST   (https://api.eduzz.com/heycamp/v1/invitations)
params: communityId, spaceId, email
# HEYCAMP: CONVITES → Listar Convites                       GET    (https://api.eduzz.com/heycamp/v1/invitations/:communityId)
path: communityId; params: page, itemsPerPage, status (accepted/pending)

# HEYCAMP: ESPAÇOS → Listar Espaços                         GET    (https://api.eduzz.com/heycamp/v1/spaces/community/:communityId)
path: communityId; params: page, itemsPerPage

# HEYCAMP: MEMBROS → Atualização de dados do membro PUT    (https://api.eduzz.com/heycamp/v1/members/:memberId)
path: memberId; params: email, socialMedias, socialMedias.instagram, socialMedias.facebook, socialMedias.linkedin, socialMedias.x
# HEYCAMP: MEMBROS → Excluir membro                         DELETE (https://api.eduzz.com/heycamp/v1/members/:memberId/community/:communityId)
path: memberId, communityId
# HEYCAMP: MEMBROS → Excluir membros em massa               DELETE (https://api.eduzz.com/heycamp/v1/members/bulk)
params: membersId (array), communityId
# HEYCAMP: MEMBROS → Listar detalhes do membro              GET    (https://api.eduzz.com/heycamp/v1/members/:memberId/community/:communityId)
path: memberId, communityId
# HEYCAMP: MEMBROS → Listar membros                         GET    (https://api.eduzz.com/heycamp/v1/members/community/:communityId)
path: communityId; params: page, itemsPerPage, startDate, endDate, status (active/deleted)
# HEYCAMP: MEMBROS → Listar membros que comentaram em posts GET    (https://api.eduzz.com/heycamp/v1/members/community/:communityId/comments)
path: communityId; params: page, itemsPerPage, days
# HEYCAMP: MEMBROS → Listar membros que curtiram posts      GET    (https://api.eduzz.com/heycamp/v1/members/community/:communityId/likes)
path: communityId; params: page, itemsPerPage, days

# HEYCAMP: POSTS → Buscar post                              GET    (https://api.eduzz.com/heycamp/v1/posts/:postId/community/:communityId)
path: communityId, postId
# HEYCAMP: POSTS → Listar posts                             GET    (https://api.eduzz.com/heycamp/v1/posts/community/:communityId)
path: communityId; params: page, itemsPerPage
# HEYCAMP: POSTS → Listar posts com mais comentários        GET    (https://api.eduzz.com/heycamp/v1/posts/community/:communityId/top/commented)
path: communityId; params: days
# HEYCAMP: POSTS → LIstar posts com mais curtidas           GET    (https://api.eduzz.com/heycamp/v1/posts/community/:communityId/top/liked)
path: communityId; params: days
# HEYCAMP: POSTS → Quantidade de comentários                GET    (https://api.eduzz.com/heycamp/v1/posts/community/:communityId/total/comments)
path: communityId; params: days

In [ ]:
# MYEDUZZ: AFILIADOS → Listagem                                          GET   (https://api.eduzz.com/myeduzz/v1/affiliates)
params: productIds, page, itemsPerPage

# MYEDUZZ: ASSINATURA → Adicionar uma cobrança adicional                 POST  (https://api.eduzz.com/myeduzz/v1/subscriptions/additional-charges)
!! Em desenvolvimento !!
# MYEDUZZ: ASSINATURA → Alterar dia de cobrança de uma assinatura        PUT   (https://api.eduzz.com/myeduzz/v1/subscriptions/:id/recurring-date)
path: id; params: dueDateDay (1/5/10/15/20/25), motive (Solicitação do cliente / Solicitação do cliente / Negociação comercial / Reorganização financeira do cliente / Mudança no método de pagamento / Problemas com a data atual de pagamento)
# MYEDUZZ: ASSINATURA → Cancelar Contratos                               POST  (https://api.eduzz.com/myeduzz/v1/subscriptions/bulk-cancel)
params: subscriptions, cancelReason
# MYEDUZZ: ASSINATURA → Detalhes de uma assinatura                       GET   (https://api.eduzz.com/myeduzz/v1/subscriptions/:id)
path: id
# MYEDUZZ: ASSINATURA → Detalhes de uma lista de assinaturas             GET   (https://api.eduzz.com/myeduzz/v1/subscriptions) ***
params: page, startDate*, endDate*, filterBy* (creation/update)

# MYEDUZZ: ASSINATURA → Detalhes do cliente                              GET   (https://api.eduzz.com/myeduzz/v1/subscriptions/customers/:email)
path: email
# MYEDUZZ: ASSINATURA → Lista as faturas de uma assinatura               GET   (https://api.eduzz.com/myeduzz/v1/subscriptions/:id/invoices)
path: id
# MYEDUZZ: ASSINATURA → Reativar Contrato                                POST  (https://api.eduzz.com/myeduzz/v1/subscriptions/:id/reactivate)
path: id
# MYEDUZZ: ASSINATURA → Solicita alteração do cartão para pagamento      PUT   (https://api.eduzz.com/myeduzz/v1/subscriptions/:id/send-card-change-email)
path: id

# MYEDUZZ: CLIENTES → Detalhes do Cliente                                GET   (https://api.eduzz.com/myeduzz/v1/customers/:email)
path: email
# MYEDUZZ: CLIENTES → Listagem dos Clientes                              GET   (https://api.eduzz.com/myeduzz/v1/customers)     ***
params: page, itemsPerPage, order (asc/desc), startDate, endDate

# MYEDUZZ: FINANCEIRO → Atualiza Status do Documento Fiscal do Afiliado  PATCH (https://api.eduzz.com/myeduzz/v1/financial/affiliate/fiscal-documents/:id/change-status)
path: id; params: status* (new, pending, waitingDocument, canceled, processed, approved, fileGenerated, processing, deleted)
# MYEDUZZ: FINANCEIRO → Atualiza Status do Documento Fiscal do Comprador PATCH (https://api.eduzz.com/myeduzz/v1/financial/buyer/fiscal-documents/:id/change-status)
path: id; params: status* (new, pending, waitingDocument, canceled, processed, approved, fileGenerated, processing, deleted)
# MYEDUZZ: FINANCEIRO → Extrato                                          GET   (https://api.eduzz.com/myeduzz/v2/financial/statement)
params: page, itemsPerPage, startDate*, endDate*, saleId
# MYEDUZZ: FINANCEIRO → Listagem dos Documentos Fiscais do Afiliado      GET   ()
params: page, itemsPerPage, startDate*, endDate*, status (new, pending, waitingDocument, canceled, processed, approved, fileGenerated, processing, deleted), name, email, saleId, productId, documentId
# MYEDUZZ: FINANCEIRO → Listagem dos Documentos FIscais do Comprador     GET
params: page, itemsPerPage, startDate*, endDate*, status (new, pending, waitingDocument, canceled, processed, approved, fileGenerated, processing, deleted), name, email, saleId, productId, documentId
# MYEDUZZ: FINANCEIRO → Transferências GET
params: page, itemsPerPage

# MYEDUZZ: PRODUTOS → Campanhas GET
path: id
# MYEDUZZ: PRODUTOS → Cupons GET
params: page, itemsPerPage
# MYEDUZZ: PRODUTOS → Detalhes GET
path: id
# MYEDUZZ: PRODUTOS → Listagem GET
params: page, itemsPerPage
# MYEDUZZ: PRODUTOS → Ofertas GET
path: id
# MYEDUZZ: PRODUTOS → Parceiros GET
path: id

# MYEDUZZ: VENDAS → Detalhes GET
path: id
# MYEDUZZ: VENDAS → Listagem GET
params: page, itemsPerPage, startDate*, endDate*, referenceDate (dueDate, paidAt, createdAt, updatedAt), contractId, status (open, processing, paid, canceled, waitingRefund, refunded, duplicated, expired, recovering, waitingPayment, scheduled, negotiated), productId, affiliateId, buyerEmail, buyerDocument
# MYEDUZZ: VENDAS → Listagem de chargebacks GET
params: page, itemsPerPage, startDate*, endDate*, id, chargebackStatus (new, won, lost, waitingDocuments, inANalysis), buyerEmail
# MYEDUZZ: VENDAS → Resumo GET
params: startDate*, endDate*, contractId, productId, affiliatedId, status (open, processing, paid, canceled, waitingRefund, refunded, duplicated, expired, recovering, waitingPayment, scheduled, negotiated)

In [ ]:
# ****************************************************************************************************************************
# MYEDUZZ: FINANCEIRO → Extrato                                          GET   (https://api.eduzz.com/myeduzz/v2/financial/statement)
# ****************************************************************************************************************************
import requests
from datetime import datetime, timedelta
start_date = (datetime.now() - timedelta(days=360)).strftime("%Y-%m-%d")
end_date   = datetime.now().strftime("%Y-%m-%d")

url = 'https://api.eduzz.com/myeduzz/v2/financial/statement'

params = {'startDate':start_date, 'endDate': end_date, 'filterBy':'creation'}
try:
    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200: print(response.json())
    else:                           print(f"❌ Erro {response.status_code}: {response.text}")
except Exception as e: print(f"Erro na conexão: {e}")

In [ ]:
# MyEduzz: Financeiro → Extrato
url = 'https://api.eduzz.com/myeduzz/v2/financial/statement'

params = {"startDate": start_date, "endDate": end_date, "page": 1, "itemsPerPage": 10}

try:
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200: print(response.json())
    else:                           print(f"❌ Erro {response.status_code}: {response.text}")

except Exception as e: print(f"Erro na conexão: {e}")

# demora um pouquinho. 642 itens

In [ ]:
url = 'https://api.eduzz.com/myeduzz/v1/financial/transfers'

params = {"startDate": start_date, "endDate": end_date, "page": 1, "itemsPerPage": 10}

try:
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200: print(response.json())
    else:                           print(f"❌ Erro {response.status_code}: {response.text}")

except Exception as e: print(f"Erro na conexão: {e}")